# Mirror Test — one-click Kaggle runner

This notebook contains the **entire pipeline inside itself** (no GitHub
needed). Every time you run it, it works out what is already done, does as
much of the remaining GPU work as fits in the session, then packages the
results into **one file** for Claude.

---

## First time only (~15 minutes of clicking)

1. **Kaggle account**: kaggle.com → sign up → *Settings → verify your phone
   number* (required before Kaggle allows GPUs + internet).
2. **Hugging Face access** (for the two gated models):
   - huggingface.co → sign up → open these two pages **while logged in** and
     click *Agree / Access repository*:
     - https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct
     - https://huggingface.co/google/gemma-2-9b-it
   - then *Settings → Access Tokens → Create new token* (type: **Read**),
     copy it.
3. **This notebook's settings** (right-hand panel ▸ Session options):
   - Accelerator: **GPU T4 x2**
   - Internet: **ON**
4. **The token secret**: top menu *Add-ons → Secrets → Add secret* →
   Label: `HF_TOKEN`, Value: the token you copied → make sure it is
   **attached** to this notebook (checkbox).

## Every run (3 clicks)

1. *(from the 2nd run onward)* right panel → **+ Add Input** → *Your Work* →
   select this notebook's **previous version output** → Add. (This is how
   the new session sees everything already computed.)
   **Safety net:** if you forget this step, the notebook automatically
   recovers all past results from your GitHub repository instead (look for
   `[seed]` lines near the top of the log) — as long as Claude has ingested
   and pushed your latest bundle. Attaching the Input is still preferred:
   it is always at least as up-to-date as GitHub.
2. Click **Save Version → Save & Run All (Commit)** → close the tab.
   Kaggle runs it in the background (up to ~8 h of work per run, then it
   stops itself safely).
3. When Kaggle notifies you it finished: open the version → **Output** tab →
   download **`mirror_bundle.zip`** and **`SESSION_REPORT.md`** → put them
   in `Desktop/Mirror/Output/` on your laptop → tell Claude *“new bundle is
   in”*.

Repeat until the report's first line says **ALL GPU WORK COMPLETE**
(expect **3–4 runs total**). Claude does all analysis and paper work from
the bundles — you never need to run anything else.

> Safe by design: every result is written incrementally with per-item
> resume, so a crashed or interrupted run costs nothing — the next run
> continues exactly where it stopped. Running this notebook twice never
> repeats finished work.


In [ ]:
# ============================== RUN SETTINGS ================================
# You normally never need to touch this cell.

# Stop starting/continuing work after this many hours, then package results.
# Kaggle GPU sessions allow ~9-12 h; 8.0 leaves a safe margin for packaging.
BUDGET_HOURS = 8.0

# Set True to only PRINT the plan/status without running any task
# (useful to check what a session would do).
DRY_RUN = False

# Safety net: if no previous session's output is attached as an Input, the
# notebook recovers past results from this PUBLIC git repository instead
# (data/ and results/ of mirror-test-llms are committed there after every
# ingested session). Set to "" to disable.
SEED_GIT_URL = "https://github.com/shehryars715/MirrorTest.git"

# Notebook build fingerprint (filled in by the builder; do not edit).
NOTEBOOK_BUILD = "2026-07-16-a27f93cc"
print(f"[config] budget={BUDGET_HOURS}h dry_run={DRY_RUN} build={NOTEBOOK_BUILD}")


In [ ]:
# =========================== BOOTSTRAP ======================================
# Unpacks the embedded pipeline code, installs dependencies, authenticates
# to Hugging Face, and pulls in everything already computed by previous
# sessions (attached as Inputs). Idempotent - safe to run every time.

import base64, io, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else \
    Path(os.environ.get("MIRROR_TEST_DIR", str(Path.home() / "mirror-run")))
REPO_DIR = WORK / "mirror-test-llms"
REPO_DIR.mkdir(parents=True, exist_ok=True)

# ---- 1. unpack the embedded repo (code + configs + tests; never data) ------
PAYLOAD_B64 = "..."
_zip_bytes = base64.b64decode(PAYLOAD_B64)
with zipfile.ZipFile(io.BytesIO(_zip_bytes)) as zf:
    zf.extractall(REPO_DIR)   # overwrites code; data/ and results/ are not in the zip
print(f"[bootstrap] pipeline code unpacked -> {REPO_DIR} "
      f"({len(_zip_bytes)//1024} KB payload)")

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))

# ---- 2. dependencies (Kaggle image has torch; top up the rest) -------------
if ON_KAGGLE:
    print("[bootstrap] installing/upgrading libraries (2-4 min) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                    "transformers", "accelerate", "bitsandbytes", "datasets",
                    "sentence-transformers", "pyyaml", "scikit-learn"],
                   check=False)

import utils  # noqa: E402  (from the unpacked src/)

# ---- 3. Hugging Face auth (Kaggle secret HF_TOKEN or env var) --------------
utils.setup_hf_auth()
HAVE_HF_TOKEN = bool(os.environ.get("HF_TOKEN"))

# ---- 4. GPU inventory -------------------------------------------------------
HAVE_GPU = False
try:
    import torch
    HAVE_GPU = torch.cuda.is_available()
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"[gpu] cuda:{i} {p.name} ({p.total_memory/2**30:.1f} GB)")
except Exception as e:  # noqa: BLE001
    print(f"[gpu] torch unavailable: {e}")
if not HAVE_GPU:
    print("[gpu] NO GPU - GPU tasks will be SKIPPED. On Kaggle: Session "
          "options -> Accelerator -> GPU T4 x2, then Save & Run All again.")

# ---- 5. seed data/results from every attached previous output --------------
def _seed_tree(base: Path) -> tuple[int, int]:
    """Copy data/ + results/ files from `base` unless a same-or-bigger local
    copy exists (JSONL outputs only ever grow, so bigger == newer)."""
    copied = kept = 0
    for sub in ("data", "results"):
        src = base / sub
        if not src.is_dir():
            continue
        for f in src.rglob("*"):
            if not f.is_file() or f.suffix == ".zip":
                continue
            dst = REPO_DIR / f.relative_to(base)
            if dst.exists() and dst.stat().st_size >= f.stat().st_size:
                kept += 1
                continue
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dst)
            copied += 1
    return copied, kept

def _candidate_bases(root: Path):
    """Places that look like a previous session's tree (possibly nested)."""
    for cand in [root, root / "mirror-test-llms"] + sorted(root.glob("*/")):
        if (cand / "data").is_dir() or (cand / "results").is_dir():
            yield cand

total_seeded = 0
inputs_root = Path("/kaggle/input")
if ON_KAGGLE:
    listed = sorted(inputs_root.iterdir()) if inputs_root.is_dir() else []
    print(f"[seed] attached Inputs: "
          f"{[p.name for p in listed] if listed else 'NONE'}")
    seen = set()
    for inp in listed:
        # a re-uploaded bundle zip? extract it to temp first
        for z in inp.rglob("mirror_bundle*.zip"):
            tmp = WORK / f"_seed_{z.stem}"
            if not tmp.exists():
                with zipfile.ZipFile(z) as zf:
                    zf.extractall(tmp)
            inp = tmp  # fall through to directory scan
        for base in _candidate_bases(inp):
            if base in seen:
                continue
            seen.add(base)
            c, k = _seed_tree(base)
            total_seeded += c + k
            print(f"[seed] {base}: copied {c} files, kept {k} local")

# ---- 6. GitHub safety net: recover past results even with no Input ---------
# The laptop-side workflow commits data/ + results/ to a public repo after
# every ingested session, so a forgotten "+ Add Input" no longer restarts
# the project from zero.
_git_url = globals().get("SEED_GIT_URL", "")
if _git_url and total_seeded == 0 and (ON_KAGGLE or os.environ.get("MIRROR_FORCE_GIT_SEED")):
    print(f"[seed] no data found in Inputs - trying git fallback: {_git_url}")
    clone_dir = WORK / "_seed_git"
    try:
        if not clone_dir.exists():
            subprocess.run(["git", "clone", "--depth", "1", _git_url,
                            str(clone_dir)], check=True, capture_output=True,
                           text=True, timeout=300)
        found = False
        for base in _candidate_bases(clone_dir):
            c, k = _seed_tree(base)
            if c or k:
                found = True
                total_seeded += c + k
                print(f"[seed] git: copied {c} files, kept {k} local from {base}")
        if not found:
            print("[seed] git repo cloned but contained no data/ or results/ yet")
    except Exception as e:  # noqa: BLE001
        print(f"[seed] git fallback failed ({e}) - continuing without it")

if ON_KAGGLE and total_seeded == 0:
    print("=" * 72)
    print("[seed] WARNING: no previous results found (no usable Input, no git")
    print("[seed] data). If this is your FIRST session, that is normal.")
    print("[seed] If it is NOT, cancel this run, click '+ Add Input' -> Your")
    print("[seed] Work -> previous version's Output, then Save & Run All.")
    print("=" * 72)

print(f"[bootstrap] ready. repo={REPO_DIR} gpu={HAVE_GPU} hf_token={HAVE_HF_TOKEN} "
      f"seeded_files={total_seeded}")


In [ ]:
# =========================== ORCHESTRATOR ===================================
# Knows the full experiment as an ordered task list with dependencies.
# Each session: evaluate what is already DONE (cheap file-count checks),
# then run the next incomplete tasks until the time budget is spent.
#
# Design note: the done-checks only steer BUDGETING - correctness never
# depends on them, because every pipeline script is internally resumable
# and instantly skips finished items. A wrong "not done" verdict just costs
# one model-load; a wrong "done" verdict is prevented by checking counts.

import json, queue, subprocess, sys, threading, time
from collections import deque
from pathlib import Path

from utils import PAIRS_DIR, GENERATIONS_DIR, JUDGMENTS_DIR, BASELINES_DIR, \
    PROMPTS_DIR, read_jsonl, load_config

CFG = load_config()
T0 = time.monotonic()
STATE_PATH = REPO_DIR / "orchestrator_state.json"
DOMAINS = ["news", "dolly", "wp"]
JUDGES = [m["key"] for m in CFG["judges"]]
FOILS = [m["key"] for m in CFG["foils"]]
GEN_FOILS = [m["key"] for m in CFG["foils"] if m.get("hf_id")]
GATED = {"llama-3.2-3b-instruct", "gemma-2-9b-it"}
BASE_JUDGES = [m["key"] for m in CFG.get("base_judges", [])]
PARA_JUDGE = CFG["paraphrase"]["judge"]
PARA_FOIL = CFG["paraphrase"]["foil"]
N_TARGET = CFG["prompt_filters"]["n_per_domain"]
PLACEBO_N = CFG["generation"]["placebo_n_prompts"]

def hours_left() -> float:
    return BUDGET_HOURS - (time.monotonic() - T0) / 3600

def _n_lines(path: Path) -> int:
    return sum(1 for line in open(path, encoding="utf-8") if line.strip()) \
        if path.exists() else 0

def n_prompts(domain: str) -> int:
    return _n_lines(PROMPTS_DIR / f"{domain}.jsonl")

# ----------------------------- done checks ----------------------------------
def prompts_done() -> bool:
    return all(n_prompts(d) >= N_TARGET for d in DOMAINS)

def gen_done(model: str, placebo: bool) -> bool:
    if not prompts_done():
        return False
    for d in DOMAINS:
        recs = read_jsonl(GENERATIONS_DIR / f"{model}__{d}.jsonl")
        s1 = sum(1 for r in recs if r.get("sample", "s1") == "s1")
        if s1 < n_prompts(d):
            return False
        if placebo:
            s2 = sum(1 for r in recs if r.get("sample") == "s2")
            if s2 < min(PLACEBO_N, n_prompts(d)):
                return False
    return True

def pairs_done() -> bool:
    if not (PAIRS_DIR / "pairs_report.json").exists():
        return False
    for j in JUDGES:
        for f in FOILS:
            for d in DOMAINS:
                if not (PAIRS_DIR / f"ppp__{j}__{f}__{d}.jsonl").exists():
                    return False
        if not (PAIRS_DIR / f"ipp__{j}.jsonl").exists():
            return False
    return True

def paraphrase_done() -> bool:
    for d in DOMAINS:
        src = _n_lines(PAIRS_DIR / f"ppp__{PARA_JUDGE}__{PARA_FOIL}__{d}.jsonl")
        if src == 0 or _n_lines(PAIRS_DIR / f"para__{PARA_JUDGE}__{PARA_FOIL}__{d}.jsonl") \
                < min(src, 200):
            return False
    return True

def _pairs_of(judge: str) -> str:
    for m in CFG.get("base_judges", []):
        if m["key"] == judge:
            return m["pairs_of"]
    return judge

def _judgment_counts(judge: str):
    recs = read_jsonl(JUDGMENTS_DIR / f"ppp__{judge}.jsonl")
    out = {"core0": 0, "placebo": 0, "para": 0, "phr12": 0}
    for r in recs:
        if r["condition"] == "core" and r["phrasing"] == 0:
            out["core0"] += 1
        elif r["condition"] == "placebo":
            out["placebo"] += 1
        elif r["condition"] == "paraphrase":
            out["para"] += 1
        if r["condition"] == "core" and r["phrasing"] in (1, 2):
            out["phr12"] += 1
    return out

def _expected_core_pairs(owner: str) -> int:
    return sum(_n_lines(PAIRS_DIR / f"ppp__{owner}__{f}__{d}.jsonl")
               for f in FOILS for d in DOMAINS)

def ppp_done(judge: str, with_placebo: bool) -> bool:
    if not pairs_done():
        return False
    owner = _pairs_of(judge)
    c = _judgment_counts(judge)
    if c["core0"] < 2 * _expected_core_pairs(owner):
        return False
    if with_placebo:
        exp_pl = sum(_n_lines(PAIRS_DIR / f"placebo__{owner}__{d}.jsonl") for d in DOMAINS)
        if c["placebo"] < 2 * exp_pl:
            return False
    return True

def main_cell_done() -> bool:
    if not paraphrase_done():
        return False
    cell = sum(_n_lines(PAIRS_DIR / f"ppp__{PARA_JUDGE}__{PARA_FOIL}__{d}.jsonl")
               for d in DOMAINS)
    passed = sum(1 for d in DOMAINS for r in
                 read_jsonl(PAIRS_DIR / f"para__{PARA_JUDGE}__{PARA_FOIL}__{d}.jsonl")
                 if r.get("passed_gate"))
    c = _judgment_counts(PARA_JUDGE)
    return c["phr12"] >= 2 * 2 * cell and c["para"] >= 2 * passed

def ipp_done(judge: str) -> bool:
    exp = _n_lines(PAIRS_DIR / f"ipp__{judge}.jsonl")
    return exp > 0 and _n_lines(JUDGMENTS_DIR / f"ipp__{judge}.jsonl") >= exp

def ppl_done(judge: str, with_para: bool) -> bool:
    owner = _pairs_of(judge)
    rows = read_jsonl(BASELINES_DIR / f"ppl__{judge}.jsonl")
    core = sum(1 for r in rows if r["condition"] == "core")
    if core < _expected_core_pairs(owner) or core == 0:
        return False
    if with_para:
        passed = sum(1 for d in DOMAINS for r in
                     read_jsonl(PAIRS_DIR / f"para__{owner}__{PARA_FOIL}__{d}.jsonl")
                     if r.get("passed_gate"))
        if sum(1 for r in rows if r["condition"] == "paraphrase") < passed:
            return False
    return True

def stylo_done() -> bool:
    if not pairs_done():
        return False
    for p in PAIRS_DIR.glob("ppp__*.jsonl"):
        if _n_lines(p) >= 20 and not \
                (BASELINES_DIR / f"stylo__{p.stem[len('ppp__'):]}.json").exists():
            return False
    return True

# ----------------------------- task table -----------------------------------
def T(tid, cmd, est_min, done_fn, after=(), gpu=True, gated=False):
    return dict(id=tid, cmd=cmd, est_min=est_min, done_fn=done_fn,
                after=list(after), gpu=gpu, gated=gated)

PY = [sys.executable]
TASKS = [T("prompts", PY + ["src/00_build_prompts.py"], 12, prompts_done, gpu=False)]
# Time estimates RECALIBRATED from session-1 measurements on Kaggle T4
# (0.5B generation measured 144 min, 1.5B 206 min - about 4x the original
# guesses; larger models extrapolated at the measured tokens/sec trend).
# Foils generate 600 items (no placebo twin), judges 1050.
GEN_EST = {"qwen2.5-0.5b-instruct": 150, "qwen2.5-1.5b-instruct": 210,
           "qwen2.5-3b-instruct": 280, "qwen2.5-7b-instruct": 450,
           "qwen2.5-14b-instruct": 900, "llama-3.2-3b-instruct": 170,
           "gemma-2-9b-it": 380, "mistral-7b-instruct-v0.3": 300}
# Generation ORDER: small judges -> foils -> big (>=10B) judges LAST.
# Rationale: the 14B is by far the most expensive model; running it last
# means the protocol-sanctioned fallback (cap the scale axis at 7B, §18)
# stays available at zero sunk cost if the schedule slips.
_params = {m["key"]: (m.get("params_b") or 0)
           for m in CFG["judges"] + CFG["foils"] if m.get("hf_id")}
GEN_PLAN = ([(m, True) for m in JUDGES if _params.get(m, 0) < 10]
            + [(m, False) for m in GEN_FOILS]
            + [(m, True) for m in JUDGES if _params.get(m, 0) >= 10])
for m, placebo in GEN_PLAN:
    cmd = PY + ["src/01_generate.py", "--models", m] + (["--placebo"] if placebo else [])
    TASKS.append(T(f"gen:{m}", cmd, GEN_EST.get(m, 200),
                   lambda m=m, p=placebo: gen_done(m, p),
                   after=["prompts"], gated=m in GATED))
ALL_GEN = [t["id"] for t in TASKS if t["id"].startswith("gen:")]
TASKS.append(T("pairs", PY + ["src/02_build_pairs.py"], 10, pairs_done,
               after=ALL_GEN, gpu=False))
TASKS.append(T("paraphrase", PY + ["src/02b_paraphrase.py"], 200, paraphrase_done,
               after=["pairs"]))
JUDGE_EST = {"qwen2.5-0.5b-instruct": 150, "qwen2.5-1.5b-instruct": 180,
             "qwen2.5-3b-instruct": 240, "qwen2.5-7b-instruct": 360,
             "qwen2.5-14b-instruct": 540}
for j in JUDGES:
    TASKS.append(T(f"ppp:{j}", PY + ["src/03_judge_ppp.py", "--judge", j,
                                     "--include-placebo"],
                   JUDGE_EST.get(j, 90), lambda j=j: ppp_done(j, True), after=["pairs"]))
TASKS.append(T("main-cell", PY + ["src/03_judge_ppp.py", "--judge", PARA_JUDGE,
                                  "--foils", PARA_FOIL, "--phrasings", "0", "1", "2",
                                  "--include-paraphrase"],
               180, main_cell_done, after=["paraphrase", f"ppp:{PARA_JUDGE}"]))
for j in JUDGES:
    TASKS.append(T(f"ipp:{j}", PY + ["src/04_judge_ipp.py", "--judge", j],
                   30, lambda j=j: ipp_done(j), after=["pairs"]))
for j in BASE_JUDGES:
    TASKS.append(T(f"ppp:{j}", PY + ["src/03_judge_ppp.py", "--judge", j],
                   90, lambda j=j: ppp_done(j, False), after=["pairs"]))
for j in JUDGES:
    para = j == PARA_JUDGE
    cmd = PY + ["src/05_baselines.py", "perplexity", "--judge", j] + \
        (["--include-paraphrase"] if para else [])
    TASKS.append(T(f"ppl:{j}", cmd, 90, lambda j=j, p=para: ppl_done(j, p),
                   after=(["paraphrase"] if para else ["pairs"])))
for j in BASE_JUDGES:
    TASKS.append(T(f"ppl:{j}", PY + ["src/05_baselines.py", "perplexity", "--judge", j],
                   90, lambda j=j: ppl_done(j, False), after=["pairs"]))
TASKS.append(T("stylometric", PY + ["src/05_baselines.py", "stylometric"], 15,
               stylo_done, after=["pairs"], gpu=False))

# ------------------------------ run loop ------------------------------------
STATUS: dict = {}
HEARTBEAT_S = 60      # print a liveness line after this many silent seconds

def clock() -> str:
    """Session clock, e.g. [0:47] = 47 min since the session started."""
    el = int(time.monotonic() - T0)
    return f"[{el // 3600}:{el % 3600 // 60:02d}]"

def save_state():
    STATE_PATH.write_text(json.dumps(
        {"build": NOTEBOOK_BUILD, "budget_hours": BUDGET_HOURS, "tasks": STATUS},
        indent=2), encoding="utf-8")

def session_progress() -> str:
    c = {}
    for v in STATUS.values():
        c[v["status"]] = c.get(v["status"], 0) + 1
    parts = " ".join(f"{k}:{v}" for k, v in sorted(c.items()))
    return f"{parts} | {hours_left():.1f} h budget left"

def run_task(t, seq: int, n_planned: int) -> str:
    """Run one pipeline script as a subprocess, streaming its output live.
    A reader thread feeds a queue so that even when the task is silent
    (model downloads / weight loading), a heartbeat line proves the session
    is alive and shows for how long it has been quiet."""
    tail = deque(maxlen=60)
    print(f"\n{clock()} ===== TASK {seq}/{n_planned}: {t['id']} "
          f"(est ~{t['est_min']} min | {hours_left():.1f} h left) =====",
          flush=True)
    print(f"{clock()} $ {' '.join(t['cmd'])}", flush=True)
    start = time.monotonic()
    proc = subprocess.Popen(t["cmd"], cwd=REPO_DIR, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    q: queue.Queue = queue.Queue()

    def _reader():
        for ln in proc.stdout:
            q.put(ln)
        q.put(None)

    threading.Thread(target=_reader, daemon=True).start()
    killed, eof = False, False
    last_output = time.monotonic()
    while not eof:
        try:
            ln = q.get(timeout=HEARTBEAT_S)
        except queue.Empty:
            quiet = time.monotonic() - last_output
            print(f"{clock()} [heartbeat] {t['id']} running "
                  f"{(time.monotonic() - start) / 60:.0f} min, no output for "
                  f"{quiet / 60:.0f} min (downloads/loading can be silent)",
                  flush=True)
        else:
            if ln is None:
                eof = True
            else:
                print(ln, end="", flush=True)
                tail.append(ln.rstrip())
                last_output = time.monotonic()
        if hours_left() <= 0 and not killed:
            print(f"\n{clock()} [budget] time is up - pausing {t['id']} "
                  "(it resumes automatically next session)", flush=True)
            proc.terminate()
            killed = True
    rc = proc.wait()
    STATUS[t["id"]]["tail"] = list(tail)[-12:]
    if killed:
        return "paused"
    if rc != 0:
        low = "\n".join(tail).lower()
        if "401" in low or "403" in low or "gated" in low or "unauthorized" in low:
            return "auth-error"
        return "failed"
    return "ran" if t["done_fn"]() else "partial"

# ---- status board + session plan --------------------------------------------
print(f"{clock()} evaluating what is already done ...", flush=True)
runnable = []
print(f"\n{'TASK':<34}{'STATE':<10}{'EST':>6}")
for t in TASKS:
    if t["done_fn"]():
        STATUS[t["id"]] = {"status": "done", "note": "already complete"}
    else:
        STATUS[t["id"]] = {"status": "todo", "note": ""}
        runnable.append(t)
    print(f"{t['id']:<34}{STATUS[t['id']]['status']:<10}"
          f"{t['est_min']:>4}m", flush=True)
todo_min = sum(t["est_min"] for t in runnable)
print(f"\n{clock()} plan: {len(TASKS) - len(runnable)}/{len(TASKS)} tasks "
      f"already done; ~{todo_min / 60:.1f} h of work remains; this session's "
      f"budget is {BUDGET_HOURS} h -> expect to run "
      f"~{min(len(runnable), max(1, round(len(runnable) * BUDGET_HOURS * 60 / max(todo_min, 1))))} "
      f"of the {len(runnable)} remaining tasks.", flush=True)
save_state()

seq = 0
for t in runnable:
    st = STATUS[t["id"]]
    deps = [d for d in t["after"] if STATUS.get(d, {}).get("status") not in
            ("done", "ran")]
    if deps:
        st.update(status="blocked", note=f"waiting on: {', '.join(deps)}")
        print(f"{clock()} [skip] {t['id']}: blocked by {', '.join(deps)}", flush=True)
    elif t["gpu"] and not HAVE_GPU:
        st.update(status="skipped", note="no GPU in this session")
        print(f"{clock()} [skip] {t['id']}: no GPU", flush=True)
    elif t["gated"] and not HAVE_HF_TOKEN:
        st.update(status="skipped",
                  note="needs HF_TOKEN secret + accepted license")
        print(f"{clock()} [skip] {t['id']}: needs HF_TOKEN secret", flush=True)
    elif hours_left() <= 0:
        st.update(status="deferred", note="session budget spent")
        print(f"{clock()} [defer] {t['id']}: budget spent", flush=True)
    elif DRY_RUN:
        st.update(status="would-run", note=f"~{t['est_min']} min")
        print(f"{clock()} [dry-run] would run {t['id']} (~{t['est_min']} min)",
              flush=True)
    else:
        seq += 1
        start = time.monotonic()
        outcome = run_task(t, seq, len(runnable))
        mins = (time.monotonic() - start) / 60
        st.update(status=outcome, note=f"{mins:.0f} min")
        print(f"{clock()} [task] {t['id']} -> {outcome.upper()} "
              f"({mins:.0f} min vs est {t['est_min']})", flush=True)
        print(f"{clock()} [session] {session_progress()}", flush=True)
    save_state()

print(f"\n{clock()} [orchestrator] pass complete - {session_progress()}")
print("finalize cell writes the report next.", flush=True)


In [ ]:
# ============================ FINALIZE ======================================
# Runs the (cheap) statistics pass on whatever exists, writes the session
# report, and packages EVERYTHING Claude needs into one zip.

import datetime, json, subprocess, sys, zipfile
from pathlib import Path

# ---- stats snapshot (harmless if too little data exists yet) ---------------
stats = subprocess.run([sys.executable, "src/06_stats.py"], cwd=REPO_DIR,
                       capture_output=True, text=True)
stats_note = ("statistics refreshed - see results/tables/"
              if stats.returncode == 0 else
              "statistics skipped (not enough judgments yet - expected in "
              "early sessions)")
print(f"[finalize] {stats_note}")

# ---- session report ---------------------------------------------------------
state = json.loads((REPO_DIR / "orchestrator_state.json").read_text(encoding="utf-8"))
tasks = state["tasks"]
order = list(tasks)
n_done = sum(1 for t in order if tasks[t]["status"] in ("done", "ran"))
todo = [t for t in order if tasks[t]["status"] in
        ("todo", "blocked", "deferred", "paused", "partial", "would-run")]
skipped = [t for t in order if tasks[t]["status"] == "skipped"]
failed = [t for t in order if tasks[t]["status"] in ("failed", "auth-error")]
EST = {t["id"]: t["est_min"] for t in TASKS}
remaining_h = sum(EST.get(t, 60) for t in todo) / 60

headline = ("ALL GPU WORK COMPLETE - give this bundle to Claude for the "
            "final analysis and paper." if not todo and not failed and not skipped
            else f"IN PROGRESS - {n_done}/{len(order)} tasks done, "
                 f"~{remaining_h:.1f} h of GPU work remaining (run this "
                 "notebook again, attaching this version's output as Input).")

def _tree_counts():
    lines = []
    for sub in ("data/prompts", "data/generations", "data/pairs",
                "results/judgments", "results/baselines", "results/tables"):
        d = REPO_DIR / sub
        n = sum(1 for f in d.rglob("*") if f.is_file()) if d.exists() else 0
        lines.append(f"| {sub} | {n} files |")
    return "\n".join(lines)

now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
rows = "\n".join(
    f"| {t} | {tasks[t]['status']} | {tasks[t].get('note','')} |" for t in order)
fail_detail = "\n".join(
    f"\n### {t} — last output lines\n```\n" + "\n".join(tasks[t].get("tail", []))
    + "\n```" for t in failed) or "none"

report = f"""# SESSION REPORT — {now}

**{headline}**

- notebook build: `{state['build']}` · budget: {state['budget_hours']} h
- GPU: {HAVE_GPU} · HF token: {HAVE_HF_TOKEN} · {stats_note}

## Task board
| task | status | note |
|---|---|---|
{rows}

`done` = complete before this session · `ran` = completed this session ·
`paused/partial` = mid-way, auto-resumes · `blocked` = waiting on earlier
tasks · `skipped` = needs GPU or HF_TOKEN (fix and re-run) ·
`deferred` = out of time this session.

## Failures needing attention
{fail_detail}

## Data inventory
| location | count |
|---|---|
{_tree_counts()}

## What to do next
1. {"Nothing on Kaggle - download mirror_bundle.zip below and hand it to Claude."
    if not todo and not failed and not skipped else
    "Re-run: open the notebook, + Add Input -> this version's Output, "
    "Save Version -> Save & Run All."}
2. {"Fix first: add the HF_TOKEN secret and accept the Llama/Gemma licenses, "
    "then re-run." if any('HF_TOKEN' in tasks[t].get('note','') for t in skipped)
    else "No settings changes needed."}
3. Download **mirror_bundle.zip** + **SESSION_REPORT.md** from this
   version's Output tab into `Desktop/Mirror/` and tell Claude.
"""

for p in (WORK / "SESSION_REPORT.md", REPO_DIR / "SESSION_REPORT.md"):
    p.write_text(report, encoding="utf-8")
print(report)

# ---- the bundle -------------------------------------------------------------
bundle = WORK / "mirror_bundle.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    for sub in ("data", "results"):
        base = REPO_DIR / sub
        if base.exists():
            for f in base.rglob("*"):
                if f.is_file() and f.suffix != ".zip":
                    zf.write(f, f.relative_to(REPO_DIR))
    for extra in ("orchestrator_state.json", "SESSION_REPORT.md"):
        if (REPO_DIR / extra).exists():
            zf.write(REPO_DIR / extra, extra)
print(f"\n[finalize] bundle ready: {bundle} "
      f"({bundle.stat().st_size/2**20:.1f} MB) - download it from the "
      "Output tab and give it to Claude.")
